In [5]:
import pandas as pd
import ast
from pathlib import Path

# Загружаем датасет из папки data
DATA_PATH = Path("data/data.csv")
df = pd.read_csv(DATA_PATH)

# Разбираем колонку artists из строки в список
def parse_artists(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return [x]

df["artists_parsed"] = df["artists"].apply(parse_artists)

# Универсальная функция выборки интересных песен
def select_interesting_songs(
    df,
    artists=None,
    year_from=None,
    year_to=None,
    min_popularity=None,
    max_popularity=None,
    min_energy=None,
    min_danceability=None,
    max_instrumentalness=None,
    explicit=None,
    n=40,
    random_state=42
):
    filtered = df.copy()

    if artists:
        filtered = filtered[
            filtered["artists_parsed"].apply(
                lambda lst: any(a in lst for a in artists)
            )
        ]

    if year_from is not None:
        filtered = filtered[filtered["year"] >= year_from]

    if year_to is not None:
        filtered = filtered[filtered["year"] <= year_to]

    if min_popularity is not None:
        filtered = filtered[filtered["popularity"] >= min_popularity]

    if max_popularity is not None:
        filtered = filtered[filtered["popularity"] <= max_popularity]

    if min_energy is not None:
        filtered = filtered[filtered["energy"] >= min_energy]

    if min_danceability is not None:
        filtered = filtered[filtered["danceability"] >= min_danceability]

    if max_instrumentalness is not None:
        filtered = filtered[filtered["instrumentalness"] <= max_instrumentalness]

    if explicit is not None:
        filtered = filtered[filtered["explicit"] == int(explicit)]

    if len(filtered) == 0:
        print("По заданным фильтрам ничего не найдено.")
        return filtered, []

    sample_size = min(n, len(filtered))
    result = filtered.sample(n=sample_size, random_state=random_state)

    return result, result.index.tolist()

# ===== НАСТРОЙ ПАРАМЕТРЫ ТУТ =====
val_df, val_indices = select_interesting_songs(
    df,
    artists=["Taylor Swift", "Drake"],  # или [] / None, если не нужен фильтр по артистам
    year_from=2010,
    year_to=2020,
    min_popularity=60,
    min_energy=0.6,
    min_danceability=0.6,
    max_instrumentalness=0.5,
    explicit=None,  # True, False или None
    n=30,           # размер валидационного набора (30–50)
    random_state=42
)

print("Индексы выбранных песен:")
print(val_indices)
print("Количество песен:", len(val_indices))

OUTPUT_PATH = "data/validation_indices.txt"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for idx in val_indices:
        f.write(f"{idx}\n")

print(f"Индексы сохранены в {OUTPUT_PATH}")

Индексы выбранных песен:
[19374, 17697, 57062, 18040, 74528, 18850, 18228, 37628, 56081, 18368, 37406, 18203, 91580, 18108, 170476, 37376, 74987, 37696, 74955, 56772, 18529, 37695, 38089, 36973, 18164, 92047, 37943, 74073, 56248, 18895]
Количество песен: 30
Индексы сохранены в data/validation_indices.txt
